# 介紹
本程式使用 pyautogui 來完成一個 Blitz 的遊戲機器人。

In [ ]:
import pyautogui as pyg
pyg.sleep(2)
basex=326 #x座標
basey=292 #y座標
gap=40 #寶石間距

In [ ]:
for row in range(8):
    for col in range(8):
        x=basex+col*gap #當前方格的x座標
        y=basey+row*gap #當前方格的y座標
        color=pyg.pixel(x,y) #(x,y)顏色
        print(row,col,x,y,color)

0 0 326 292 (24, 15, 19)
0 1 366 292 (24, 16, 19)
0 2 406 292 (25, 17, 19)
0 3 446 292 (25, 17, 20)
0 4 486 292 (25, 17, 20)
0 5 526 292 (26, 18, 20)
0 6 566 292 (20, 7, 29)
0 7 606 292 (29, 16, 40)
1 0 326 332 (35, 16, 46)
1 1 366 332 (24, 7, 33)
1 2 406 332 (35, 17, 46)
1 3 446 332 (24, 7, 33)
1 4 486 332 (35, 17, 46)
1 5 526 332 (24, 7, 33)
1 6 566 332 (35, 16, 46)
1 7 606 332 (24, 7, 33)
2 0 326 372 (253, 0, 21)
2 1 366 372 (253, 232, 24)
2 2 406 372 (31, 9, 39)
2 3 446 372 (44, 19, 53)
2 4 486 372 (31, 9, 39)
2 5 526 372 (43, 19, 53)
2 6 566 372 (31, 9, 39)
2 7 606 372 (253, 7, 30)
3 0 326 412 (55, 27, 63)
3 1 366 412 (167, 167, 167)
3 2 406 412 (7, 133, 253)
3 3 446 412 (167, 167, 167)
3 4 486 412 (214, 48, 214)
3 5 526 412 (5, 220, 28)
3 6 566 412 (7, 133, 253)
3 7 606 412 (245, 28, 56)
4 0 326 452 (57, 29, 58)
4 1 366 452 (157, 3, 157)
4 2 406 452 (73, 253, 112)
4 3 446 452 (233, 233, 233)
4 4 486 452 (57, 29, 58)
4 5 526 452 (73, 42, 74)
4 6 566 452 (229, 229, 229)
4 7 606 452

In [ ]:
def get_gem_position(r,c):
    x=basex+c*gap
    y=basey+r*gap
    return(x,y) #傳回座標

In [ ]:
def get_gem_color(r,c,im):
    x,y=get_gem_position(r,c)
    return im.getpixel((x,y))

In [ ]:
def can_move_left(r,c,im):
    # 僅檢查水平向左移動 (r, c) -> (r, c-1)
    # 明確排除對角線移動
    if not (0 <= r < 8 and 1 <= c < 8):
    # 行必須在 0-7，欄必須能向左移動 (c >= 1)
        return False

    # 檢查 (r, c-1) 與 (r, c) 交換後是否形成三消
    # 只檢查同一行 (水平方向)，不檢查對角線
    left_color = get_gem_color(r, c-1, im)
    right_color = get_gem_color(r, c, im)
    
    # 檢查以 (r, c-1) 為起點的水平三連
    if left_color == right_color:
        # 再檢查旁邊一格 (c-2) 是否相同顏色
        if c >= 2:
            left2_color = get_gem_color(r, c-2, im)
            if left_color == left2_color:
                return True
    
    # 檢查以 (r, c) 為起點的水平三連 (包含左邊)
    # 情況: (r, c-1), (r, c), (r, c+1) 全相同 (若 c+1 存在)
    # 或者 (r, c), (r, c+1), (r, c+2) 全相同
    
    # 檢查右側兩格是否相同
    if c <= 5:  # 確保 c+2 在 8x8 板內
        right1 = get_gem_color(r, c+1, im)
        right2 = get_gem_color(r, c+2, im)
        if right_color == right1 == right2:
            return True
    
    return False # 若所有水平組合均無法形成三消，回傳 False

In [ ]:
def can_move_right(r, c, im): # 判斷位於 (r, c) 的寶石是否可以向右移動與 (r, c+1) 交換並形成三連消除
    # 僅檢查水平向右移動 (r, c) -> (r, c+1)
    # 明確排除對角線移動
    if not (0 <= r < 8 and 0 <= c < 7): # 行在 0-7，欄能向右移動 (c <= 6)
        return False

    left_color = get_gem_color(r, c, im)
    right_color = get_gem_color(r, c+1, im)
    
    # 檢查以 (r, c+1) 為起點的水平三連
    if right_color == left_color:
    # 情況: (r, c), (r, c+1), (r, c+2) 全相同
        if c <= 5:  # 確保 c+2 在 8x8 板內
            right1 = get_gem_color(r, c+2, im)
            if right_color == right1:
                # 再檢查 c+3
                if c <= 6:
                    right2 = get_gem_color(r, c+3, im)
                    if right_color == right1 == right2:
                        return True
                else:
                    return True  # 三格足夠
    
    # 檢查以 (r, c) 為起點的水平三連
    # 情況: (r, c-1), (r, c), (r, c+1) 全相同 (若存在)
    if c >= 1:
        left1 = get_gem_color(r, c-1, im)
        if left_color == left1:
            # 再檢查 c-2
            if c >= 2:
                left2 = get_gem_color(r, c-2, im)
                if left_color == left1 == left2:
                    return True
            else:
                return True  # 三格足夠
    
    return False # 若所有水平組合均無法形成三消，回傳 False

In [ ]:
def can_move_up(r, c, im): # 向上移動判斷 (r 需要在 1 到 7 之間)
    if not (1 <= r < 8 and 0 <= c < 8):
        return False

    top_color = get_gem_color(r-1, c, im)
    bottom_color = get_gem_color(r, c, im)

    def color_after_swap(row, col):
        if (row, col) == (r-1, c):
            return bottom_color
        if (row, col) == (r, c):
            return top_color
        return get_gem_color(row, col, im)

    for row, col in ((r-1, c), (r, c)):
        for dr, dc in ((0, 1), (1, 0)):
            for offset in (-2, -1, 0):
                cells = [(row + dr * (offset + i), col + dc * (offset + i)) for i in range(3)]
                if all(0 <= rr < 8 and 0 <= cc < 8 for rr, cc in cells):
                    colors = [color_after_swap(rr, cc) for rr, cc in cells]
                    if colors[0] == colors[1] == colors[2]:
                        return True
    return False


def can_move_down(r, c, im): # 向下移動判斷 (r 需要在 0 到 6 之間)
    if not (0 <= r < 7 and 0 <= c < 8):
        return False

    top_color = get_gem_color(r, c, im)
    bottom_color = get_gem_color(r+1, c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return bottom_color
        if (row, col) == (r+1, c):
            return top_color
        return get_gem_color(row, col, im)

    for row, col in ((r, c), (r+1, c)):
        for dr, dc in ((0, 1), (1, 0)):
            for offset in (-2, -1, 0):
                cells = [(row + dr * (offset + i), col + dc * (offset + i)) for i in range(3)]
                if all(0 <= rr < 8 and 0 <= cc < 8 for rr, cc in cells):
                    colors = [color_after_swap(rr, cc) for rr, cc in cells]
                    if colors[0] == colors[1] == colors[2]:
                        return True
    return False

In [ ]:
import pyautogui as pyg
import time

start_time = time.time()
LIMIT_TIME = 60.0  # 限時 60 秒

im = pyg.screenshot()

click_history = {}      # 防重複點擊 3 秒冷卻
fail_state_history = {} # 記錄 (座標, 寶石類別) 的失敗次數

COOLDOWN = 3.0  

no_move_count = 0  
last_print_time = time.time()

# 根據 RGB 判定寶石的「真實顏色分類」，徹底免疫閃光干擾！
def classify_gem_color(r, c, img):
    x, y = get_gem_position(r, c)
    rgb = img.getpixel((x, y))[:3]
    red, green, blue = rgb[0], rgb[1], rgb[2]
    
    # 判斷大致色相
    if red > 200 and green > 200 and blue > 200:
        return "WHITE"
    elif red > 200 and green > 150 and blue < 80:
        return "YELLOW"
    elif red > 200 and green < 100 and blue < 100:
        return "RED"
    elif green > 180 and red < 150:
        return "GREEN"
    elif blue > 180 and red < 150:
        return "BLUE"
    elif red > 180 and blue > 180:
        return "PURPLE"
    elif red > 200 and green > 100 and blue < 80:
        return "ORANGE"
    return "UNKNOWN"

print("🤖 免疫閃光機器人啟動！（以寶石類別識別，絕不上當卡死）")

while True:
    current_time = time.time()
    elapsed = current_time - start_time
    
    #  滿 60 秒精確自動停止
    if elapsed >= LIMIT_TIME:
        print(f"⏰ 滿 {LIMIT_TIME:.0f} 秒！遊戲順利結束！")
        break

    moved = False

    #  精確色彩掃描
    for r in range(8):
        for c in range(8):

            # 起點冷卻中，跳過
            if current_time - click_history.get((r, c), 0) < COOLDOWN:
                continue

            target_r, target_c = None, None

            if can_move_left(r, c, im):
                target_r, target_c = r, c - 1
            elif can_move_right(r, c, im):
                target_r, target_c = r, c + 1
            elif can_move_up(r, c, im):
                target_r, target_c = r - 1, c
            elif can_move_down(r, c, im):
                target_r, target_c = r + 1, c

            # 終點冷卻中，跳過
            if target_r is not None and (current_time - click_history.get((target_r, target_c), 0) < COOLDOWN):
                target_r, target_c = None, None

            # 找到組合
            if target_r is not None:
                # 取得兩顆寶石的「顏色類別」而非波動的 RGB 數值
                cat1 = classify_gem_color(r, c, im)
                cat2 = classify_gem_color(target_r, target_c, im)
                
                # 綁定「座標 + 寶石顏色類別」
                state_key = (
                    tuple(sorted([(r, c), (target_r, target_c)])),
                    tuple(sorted([cat1, cat2]))
                )
                
                # 同一組顏色在同一位置嘗試失敗 2 次，
                if fail_state_history.get(state_key, 0) >= 2:
                    continue

                x1, y1 = get_gem_position(r, c)
                x2, y2 = get_gem_position(target_r, target_c)

                pyg.moveTo(x1, y1)
                pyg.dragTo(x2, y2, duration=0.12, button='left')

                # 記錄冷卻與失敗次數
                click_history[(r, c)] = current_time
                click_history[(target_r, target_c)] = current_time
                
                fail_state_history[state_key] = fail_state_history.get(state_key, 0) + 1
                
                moved = True
                break

        if moved:
            break

    # 3. 狀態處置
    if moved:
        no_move_count = 0 
        time.sleep(0.65)   # 給予寶石掉落與連爆時間
        im = pyg.screenshot()
    else:
        no_move_count += 1
        time.sleep(0.15)
        im = pyg.screenshot()

        # 連續 2 圈全盤找不到解：解鎖冷卻，繼續尋找
        if no_move_count >= 2:
            click_history.clear()
            no_move_count = 0

    # 每 3 秒印出 log
    if current_time - last_print_time > 3:
        remaining = LIMIT_TIME - elapsed
        print(f"⌛ 已過 {elapsed:.1f} 秒 | 剩餘 {remaining:.1f} 秒")
        last_print_time = current_time

🤖 免疫閃光機器人啟動！（以寶石類別識別，絕不上當卡死）
⌛ 已過 3.3 秒 | 剩餘 56.7 秒
⌛ 已過 6.4 秒 | 剩餘 53.6 秒
⌛ 已過 9.4 秒 | 剩餘 50.6 秒
⌛ 已過 12.6 秒 | 剩餘 47.4 秒
⌛ 已過 15.7 秒 | 剩餘 44.3 秒
⌛ 已過 18.8 秒 | 剩餘 41.2 秒
⌛ 已過 21.9 秒 | 剩餘 38.1 秒
⌛ 已過 24.9 秒 | 剩餘 35.1 秒
⌛ 已過 28.9 秒 | 剩餘 31.1 秒
⌛ 已過 32.1 秒 | 剩餘 27.9 秒
⌛ 已過 35.4 秒 | 剩餘 24.6 秒
⌛ 已過 38.7 秒 | 剩餘 21.3 秒
⌛ 已過 42.1 秒 | 剩餘 17.9 秒
⌛ 已過 45.3 秒 | 剩餘 14.7 秒
⌛ 已過 48.6 秒 | 剩餘 11.4 秒
⌛ 已過 51.8 秒 | 剩餘 8.2 秒
⌛ 已過 55.2 秒 | 剩餘 4.8 秒
⌛ 已過 58.4 秒 | 剩餘 1.6 秒
⏰ 滿 60 秒！遊戲順利結束！


In [ ]:
# 取得指定 (r, c) 寶石中心區域 (5x5 像素) 的平均 RGB 顏色，大幅降低閃光與漸層誤判
def get_gem_color(r, c, im):
    x, y = get_gem_position(r, c)
    
    r_total, g_total, b_total = 0, 0, 0
    count = 0
    
    # 採樣中心點周圍 -2 到 +2 像素 (5x5 共 25 個點)
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            # 確保座標在圖片範圍內
            px, py = int(x + dx), int(y + dy)
            if 0 <= px < im.width and 0 <= py < im.height:
                r_val, g_val, b_val = im.getpixel((px, py))[:3]
                r_total += r_val
                g_total += g_val
                b_total += b_val
                count += 1
                
    # 傳回平均 RGB 顏色
    return (r_total // count, g_total // count, b_total // count)

In [ ]:
# 判斷兩種 RGB 平均顏色是否足夠接近 (預設容錯 tolerance = 25)
def is_color_similar(c1, c2, tolerance=25):
    return abs(c1[0] - c2[0]) < tolerance and \
           abs(c1[1] - c2[1]) < tolerance and \
           abs(c1[2] - c2[2]) < tolerance

In [ ]:
# 以 can_move_left 為例：
def can_move_left(r, c, im):
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    left_color = get_gem_color(r, c-1, im)
    right_color = get_gem_color(r, c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c-1):
            return right_color
        if (row, col) == (r, c):
            return left_color
        return get_gem_color(row, col, im)

    for row, col in ((r, c-1), (r, c)):
        for dr, dc in ((0, 1), (1, 0)):
            for offset in (-2, -1, 0):
                cells = [(row + dr * (offset + i), col + dc * (offset + i)) for i in range(3)]
                if all(0 <= rr < 8 and 0 <= cc < 8 for rr, cc in cells):
                    colors = [color_after_swap(rr, cc) for rr, cc in cells]
                    # 使用相近色判斷代替精確相等比對
                    if is_color_similar(colors[0], colors[1]) and is_color_similar(colors[1], colors[2]):
                        return True
    return False